In [0]:
%sql
-- LEAD_MERGE_MAP — CRM lead-merge losers → surviving winner records.
-- BUILT 11/19/2026 after the merge-ghost measurement: silver has NO tombstone
-- policy, so a lead the CRM merges away keeps its row forever. Measured 11/19:
-- 138 merge pairs; 85 losers have no surviving lead row, 53 still exist —
-- including 2 still status_label='Coaching Client' (same person as their
-- winners: the CC book is 1,390 leads = 1,388 persons).
-- SOURCE: silver LeadMerge activities (type='LeadMerge', merge_status=
-- 'completed') — loser = source_lead_id, winner = lead_id. Chain-resolved to
-- the ROOT winner (a loser can itself be a winner; 1 chain observed 11/19).
-- POLICY: this is a MAP, not a delete. Ghost rows stay in the spine (the
-- all-lead grain never deletes — a delete would be resurrected by any
-- rebuild-from-bronze anyway). Consumers fold duplicates through
-- COALESCE(merged_into_lead_id, lead_id) — lead_dim (3_04) carries the column.
-- Cycle guard: if chain-walking lands back on the loser (a→b→a), fall back
-- to the direct winner so the map never self-references.
CREATE OR REPLACE TABLE crm_ingestion.gold.lead_merge_map AS (
WITH raw_pairs AS (
    SELECT
        source_lead_id AS loser_lead_id,
        lead_id        AS direct_winner_lead_id,
        MAX(activity_at) AS merged_at,
        COUNT(*)        AS merge_events
    FROM crm_ingestion.silver.lead_activites_processed
    WHERE type = 'LeadMerge'
      AND merge_status = 'completed'
      AND source_lead_id IS NOT NULL
      AND source_lead_id <> lead_id
    GROUP BY source_lead_id, lead_id
),
-- chain resolution: 3 hops covers any merge chain in this data
-- (observed max depth 2: loser → winner → winner's winner)
h1 AS (
    SELECT r.*, rp.direct_winner_lead_id AS hop1_winner
    FROM raw_pairs r
    LEFT JOIN raw_pairs rp ON r.direct_winner_lead_id = rp.loser_lead_id
),
h2 AS (
    SELECT h1.*, rp2.direct_winner_lead_id AS hop2_winner
    FROM h1
    LEFT JOIN raw_pairs rp2 ON COALESCE(h1.hop1_winner, h1.direct_winner_lead_id) = rp2.loser_lead_id
),
h3 AS (
    SELECT h2.*, rp3.direct_winner_lead_id AS hop3_winner
    FROM h2
    LEFT JOIN raw_pairs rp3 ON COALESCE(h2.hop2_winner, h2.hop1_winner, h2.direct_winner_lead_id) = rp3.loser_lead_id
)
SELECT
    loser_lead_id,
    direct_winner_lead_id,
    -- cycle guard: never map a lead onto itself
    CASE WHEN COALESCE(hop3_winner, hop2_winner, hop1_winner, direct_winner_lead_id) = loser_lead_id
         THEN direct_winner_lead_id
         ELSE COALESCE(hop3_winner, hop2_winner, hop1_winner, direct_winner_lead_id)
    END AS root_lead_id,
    CASE WHEN hop3_winner IS NOT NULL THEN 3
         WHEN hop2_winner IS NOT NULL THEN 2
         WHEN hop1_winner IS NOT NULL THEN 1
         ELSE 0 END AS hops,
    merged_at,
    merge_events
FROM h3
)

In [0]:
%sql
-- MERGE-MAP NIGHTLY REPORT — the operator line. Self-contained (reads silver
-- + the map only, so numbers are current even mid-DAG before lead_dim
-- rebuilds). ghost_leads_in_cc_book > 0 = duplicate persons in the client
-- book; new value overnight = a fresh CRM merge created a new ghost.
WITH latest AS (
    SELECT id AS lead_id, status_label
    FROM (
        SELECT id, status_label,
               ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS rn
        FROM crm_ingestion.silver.leads_processed
    )
    WHERE rn = 1
),
cc AS (
    SELECT l.lead_id, COALESCE(m.root_lead_id, l.lead_id) AS person_id
    FROM latest l
    LEFT JOIN crm_ingestion.gold.lead_merge_map m ON l.lead_id = m.loser_lead_id
    WHERE l.status_label = 'Coaching Client'
),
s AS (
    SELECT
        (SELECT COUNT(*) FROM crm_ingestion.gold.lead_merge_map)                AS merge_pairs,
        (SELECT COUNT(*) FROM crm_ingestion.gold.lead_merge_map WHERE hops > 0) AS chain_resolved_pairs,
        (SELECT COUNT(*) FROM cc)                                                AS cc_lead_count,
        (SELECT COUNT(DISTINCT person_id) FROM cc)                              AS cc_person_count
)
SELECT merge_pairs, chain_resolved_pairs, cc_lead_count, cc_person_count,
       cc_lead_count - cc_person_count AS ghost_leads_in_cc_book
FROM s